# SHAP Explorer

notebook นี้ **ไม่เทรนโมเดล** — โหลดโมเดลที่ `03_train_multiclass.py` เทรนไว้แล้วมาเล่นกราฟ

ต้องรันตามลำดับนี้ก่อน:
```
python 01_prepare_data.py
python 03_train_multiclass.py
```

เลือก kernel เป็น `.venv` ที่มุมขวาบนของ VS Code ก่อนรัน

In [ ]:
import joblib
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt

import common, config

df = common.load_clean()
attacks = df[df['binary_label'] == 1].reset_index(drop=True)

model = joblib.load(config.MODEL_DIR / 'multi_best.pkl')
le = joblib.load(config.MODEL_DIR / 'label_encoder.pkl')

X = attacks[common.feature_columns(attacks)]
X_sample = X.sample(n=min(config.SHAP_SAMPLE, len(X)), random_state=config.SEED)

print(type(model).__name__, '|', len(le.classes_), 'classes:', list(le.classes_))
print('sample:', X_sample.shape)

In [ ]:
explainer = shap.TreeExplainer(model)
sv = common.to_shap_array(explainer.shap_values(X_sample))
print('shape =', sv.shape, ' (samples, features, classes)')

## 1. ภาพรวม — ฟีเจอร์ไหนสำคัญกับคลาสไหน

แท่งซ้อนกัน แต่ละสีคือหนึ่งคลาส ยิ่งยาว = ฟีเจอร์นั้นมีผลกับการตัดสินใจมาก

In [ ]:
sv_list = [sv[:, :, i] for i in range(sv.shape[2])]
shap.summary_plot(sv_list, X_sample, class_names=list(le.classes_),
                  plot_type='bar', max_display=20, show=False)
plt.tight_layout()
plt.savefig(config.FIG_DIR / 'shap_bar_all_classes.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Beeswarm ของคลาสเดียว — วิธีอ่านกราฟที่คุณสับสน

| องค์ประกอบ | ความหมาย |
|---|---|
| **ลำดับแกน Y** | เรียงตาม mean(\|SHAP\|) มาก→น้อย = ความสำคัญตัวจริง |
| **ตำแหน่งแกน X** | ค่า SHAP ของ *แต่ละแถวข้อมูล* หน่วยเป็น log-odds ไม่ใช่ probability<br>ขวา = ดันไปทาง "เป็นคลาสนี้" / ซ้าย = ดันออกจากคลาสนี้ |
| **สี** | ค่าดิบของฟีเจอร์นั้น — <span style="color:red">แดง = สูง</span>, <span style="color:blue">น้ำเงิน = ต่ำ</span> |
| **การกระจายแนวตั้ง** | jitter ล้วน ๆ **ไม่มีความหมายเชิงตัวเลข** แค่กันจุดทับกัน<br>แต่ *ความหนา* ของแถบบอกว่าจุดกระจุกตรงนั้นเยอะ |

**อ่านยังไงให้ได้ข้อสรุป:**
- แดงกองขวา / น้ำเงินกองซ้าย → ค่าฟีเจอร์ยิ่งสูง ยิ่งเป็นคลาสนี้ (ความสัมพันธ์ทางเดียว ตีความง่าย)
- แดงกระจายทั้งสองฝั่ง → มี interaction กับฟีเจอร์อื่น ต้องไปดู dependence plot ต่อ (ข้อ 3)
- จุดกระจุกที่ SHAP≈0 เกือบหมด มีหางยาวไม่กี่จุด → ฟีเจอร์นี้สำคัญเฉพาะบางกรณี

In [ ]:
CLASS = 'DDoS'   # <-- เปลี่ยนชื่อคลาสตรงนี้แล้วรันใหม่

idx = list(le.classes_).index(CLASS)
shap.summary_plot(sv[:, :, idx], X_sample, max_display=20, show=False)
plt.title(f'SHAP summary — {CLASS}')
plt.tight_layout()
plt.savefig(config.FIG_DIR / f'shap_beeswarm_{CLASS}.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Dependence plot — ดูว่าฟีเจอร์เดียวส่งผลยังไง

แกน X = ค่าดิบของฟีเจอร์, แกน Y = SHAP ของฟีเจอร์นั้น
ถ้าเส้นแนวโน้มชัดเจน = ความสัมพันธ์ตรงไปตรงมา  ถ้ากระจายเป็นแถบ = มี interaction

In [ ]:
imp = common.mean_abs_shap(sv, list(X_sample.columns), list(le.classes_))
FEATURE = imp[CLASS].idxmax()   # ฟีเจอร์อันดับ 1 ของคลาสนี้

shap.dependence_plot(FEATURE, sv[:, :, idx], X_sample, show=False)
plt.tight_layout()
plt.savefig(config.FIG_DIR / f'shap_dependence_{CLASS}.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. เทียบฟีเจอร์สำคัญข้ามคลาส

ตารางนี้คือต้นตอของปัญหาที่อาจารย์ถาม: แต่ละคลาสได้ฟีเจอร์ไม่เหมือนกัน
ค่าถูก normalize ให้แต่ละคอลัมน์รวมกันได้ 1 แล้ว จึงเทียบข้ามคลาสได้

In [ ]:
top_union = set()
for c in imp.columns:
    top_union |= set(imp[c].nlargest(10).index)

view = imp.loc[sorted(top_union)].sort_values(imp.columns[0], ascending=False)
display(view.style.background_gradient(axis=0, cmap='Reds').format('{:.4f}'))

In [ ]:
# ฟีเจอร์ที่ติด Top-10 ของกี่คลาส — ตัวที่ติดทุกคลาสคือ 'ฟีเจอร์ร่วม'
counts = pd.Series(0, index=imp.index)
for c in imp.columns:
    counts[imp[c].nlargest(10).index] += 1
counts[counts > 0].sort_values(ascending=False).to_frame('ติด Top-10 กี่คลาส')

## 5. Waterfall — อธิบายการทำนายทีละ 1 flow

ใช้ตอนต้องอธิบายให้คนอื่นฟังว่า "ทำไม flow นี้ถึงถูกตัดสินว่าเป็น attack"
E[f(x)] คือค่าเฉลี่ยของ output ทั้ง dataset แล้วแต่ละฟีเจอร์ดันขึ้น/ลงจนได้ f(x) สุดท้าย

In [ ]:
ROW = 0   # เปลี่ยนเลขแถวได้

base = explainer.expected_value
base = base[idx] if np.ndim(base) > 0 else base

shap.plots.waterfall(
    shap.Explanation(values=sv[ROW, :, idx], base_values=base,
                     data=X_sample.iloc[ROW].values,
                     feature_names=list(X_sample.columns)),
    max_display=15, show=False)
plt.tight_layout()
plt.savefig(config.FIG_DIR / f'shap_waterfall_{CLASS}_row{ROW}.png', dpi=150, bbox_inches='tight')
plt.show()